# Step Counter - Hyperparameter Search (Google Colab)

**Setup Instructions:**
1. Open the notebook in Google Colab
2. Runtime to GPU (T4)
3. Upload the windowed preprocessed data to Google Drive: `/path_to_your_project/data/processed/`
4. Make sure the notebook script is able to find the data
5. Configure hyperparameter grid below and run all cells

**Features:**
- Loop through multiple hyperparameter combinations
- Each run saves to its own experiment folder
- All results tracked in experiments.csv for

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime
import csv
from itertools import product

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Set device - Colab GPU!
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## Hyperparameter Search Configuration

Define the search space here. The script will train a model for each combination.

In [ ]:
# Base configuration
MODEL_TYPE = 'deep_cnn'  # Options: 'shallow_cnn', 'deep_cnn'
TASK_TYPE = 'regression'  # Options: 'binary', 'regression'

# Google Drive paths
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/Colab_Notebooks/Step_counter_project/data/processed')
DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/Colab_Notebooks/Step_counter_project/models/saved')

# Hyperparameter search grid - add/remove parameters as needed
HYPERPARAMETER_GRID = {
    'batch_size': [32, 64, 128, 256],
    'learning_rate': [0.001],
    'n_filters': [64],
    'dropout_rate': [0.3]
}

# Fixed training parameters
FIXED_PARAMS = {
    'epochs': 50,
    'patience': 10
}

# Generate all combinations
param_names = list(HYPERPARAMETER_GRID.keys())
param_values = list(HYPERPARAMETER_GRID.values())
all_combinations = list(product(*param_values))

print(f'Model: {MODEL_TYPE}')
print(f'Task: {TASK_TYPE}')
print(f'\nHyperparameter Grid:')
for key, values in HYPERPARAMETER_GRID.items():
    print(f'  {key}: {values}')
print(f'\nTotal experiments to run: {len(all_combinations)}')
print(f'\nAll combinations:')
for i, combo in enumerate(all_combinations, 1):
    combo_dict = dict(zip(param_names, combo))
    print(f'  {i}. {combo_dict}')

## Load Data Once

Load data before the loop to avoid repeated I/O

In [ ]:
# Load data
label_key = 'y_binary' if TASK_TYPE == 'binary' else 'y_count'

train_data = np.load(DRIVE_DATA_DIR / 'cnn_train_data.npz')
X_train = train_data['X']
y_train = train_data[label_key].astype(np.float32)

val_data = np.load(DRIVE_DATA_DIR / 'cnn_val_data.npz')
X_val = val_data['X']
y_val = val_data[label_key].astype(np.float32)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train).permute(0, 2, 1)
y_train_tensor = torch.FloatTensor(y_train)
X_val_tensor = torch.FloatTensor(X_val).permute(0, 2, 1)
y_val_tensor = torch.FloatTensor(y_val)

# Create datasets (DataLoaders will be created in the loop with varying batch_size)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

print(f'✓ Data loaded')
print(f'  Train: {len(X_train)} samples')
print(f'  Val: {len(X_val)} samples')
print(f'  Input shape: {X_train.shape[1:]} (time_steps, channels)')

## Define Model Architectures

In [ ]:
class ShallowCNN(nn.Module):
    """Shallow 1D CNN with 1 conv layer + dense layers"""
    
    def __init__(self, input_channels, sequence_length, num_filters=64, kernel_size=5, pool_size=2,
                 output_type='binary', dropout_rate=0.5):
        super(ShallowCNN, self).__init__()
        self.output_type = output_type

        self.conv1 = nn.Conv1d(input_channels, num_filters, kernel_size, padding=kernel_size // 2)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(pool_size)
        self.dropout = nn.Dropout(dropout_rate)

        conv_output_length = sequence_length // pool_size
        flattened_size = num_filters * conv_output_length

        self.linear1 = nn.Linear(flattened_size, 128)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.linear2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        x = self.linear2(x)

        if self.output_type == 'binary':
            x = self.sigmoid(x)

        return x.squeeze(-1)


class DeepCNN(nn.Module):
    """Deep 1D CNN with 5 conv blocks + batch norm + global pooling"""
    
    def __init__(self, input_channels, sequence_length, num_filters=64, output_type='binary', dropout_rate=0.5):
        super(DeepCNN, self).__init__()
        self.output_type = output_type

        # Conv Block 1
        self.conv1 = nn.Conv1d(input_channels, num_filters, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(num_filters)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.dropout_conv1 = nn.Dropout(dropout_rate * 0.5)

        # Conv Block 2
        self.conv2 = nn.Conv1d(num_filters, num_filters * 2, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(num_filters * 2)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.dropout_conv2 = nn.Dropout(dropout_rate * 0.5)

        # Conv Block 3
        self.conv3 = nn.Conv1d(num_filters * 2, num_filters * 4, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(num_filters * 4)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.dropout_conv3 = nn.Dropout(dropout_rate * 0.5)

        # Conv Block 4
        self.conv4 = nn.Conv1d(num_filters * 4, num_filters * 8, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm1d(num_filters * 8)
        self.pool4 = nn.MaxPool1d(kernel_size=2)
        self.dropout_conv4 = nn.Dropout(dropout_rate * 0.5)

        # Conv Block 5
        self.conv5 = nn.Conv1d(num_filters * 8, num_filters * 8, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm1d(num_filters * 8)
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        # Dense layers
        self.dense1 = nn.Linear(num_filters * 8, 256)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dense2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.dense3 = nn.Linear(128, 64)
        self.dropout3 = nn.Dropout(dropout_rate)
        self.output = nn.Linear(64, 1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Conv blocks with batch norm
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.dropout_conv1(self.pool1(x))

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.dropout_conv2(self.pool2(x))

        x = self.relu(self.bn3(self.conv3(x)))
        x = self.dropout_conv3(self.pool3(x))

        x = self.relu(self.bn4(self.conv4(x)))
        x = self.dropout_conv4(self.pool4(x))

        x = self.relu(self.bn5(self.conv5(x)))
        x = self.global_pool(x).squeeze(-1)

        # Dense layers
        x = self.relu(self.dense1(x))
        x = self.dropout1(x)
        x = self.relu(self.dense2(x))
        x = self.dropout2(x)
        x = self.relu(self.dense3(x))
        x = self.dropout3(x)
        x = self.output(x)

        if self.output_type == 'binary':
            x = self.sigmoid(x)

        return x.squeeze(-1)


print('✓ Models defined: ShallowCNN, DeepCNN')

## Training Function

In [ ]:
def train_model(model, train_loader, val_loader, config):
    """Train model with early stopping"""
    
    model = model.to(device)
    criterion = nn.MSELoss() if config['task_type'] == 'regression' else nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=False)
    
    history = {'loss': [], 'val_loss': [], 'mae': [], 'val_mae': []}
    best_val_loss = float('inf')
    patience_counter = 0
    best_epoch = 0
    
    for epoch in range(config['epochs']):
        # Train
        model.train()
        train_loss, train_preds, train_targets = 0, [], []
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * X_batch.size(0)
            train_preds.extend(outputs.detach().cpu().numpy())
            train_targets.extend(y_batch.cpu().numpy())
        
        train_loss /= len(train_loader.dataset)
        train_mae = np.abs(np.array(train_preds) - np.array(train_targets)).mean()
        
        # Validate
        model.eval()
        val_loss, val_preds, val_targets = 0, [], []
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                val_loss += loss.item() * X_batch.size(0)
                val_preds.extend(outputs.cpu().numpy())
                val_targets.extend(y_batch.cpu().numpy())
        
        val_loss /= len(val_loader.dataset)
        val_mae = np.abs(np.array(val_preds) - np.array(val_targets)).mean()
        
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['mae'].append(train_mae)
        history['val_mae'].append(val_mae)
        
        # Only print every 10 epochs to reduce output
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/{config["epochs"]} - loss: {train_loss:.4f} - mae: {train_mae:.4f} - val_loss: {val_loss:.4f} - val_mae: {val_mae:.4f}')
        
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), config['experiment_dir'] / 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                print(f'  Early stopping at epoch {epoch+1}')
                break
    
    # Load best model
    model.load_state_dict(torch.load(config['experiment_dir'] / 'best_model.pth'))
    history['best_epoch'] = best_epoch
    
    return history, model


def evaluate_model(model, val_loader, config):
    """Evaluate model and return metrics"""
    
    model.eval()
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X).squeeze()
            val_preds.extend(outputs.cpu().numpy())
            val_targets.extend(batch_y.numpy())
    
    val_preds = np.array(val_preds)
    val_targets = np.array(val_targets)
    
    # Calculate metrics
    val_mae = np.abs(val_preds - val_targets).mean()
    val_rmse = np.sqrt(((val_preds - val_targets) ** 2).mean())
    val_r2 = 1 - (np.sum((val_targets - val_preds) ** 2) / np.sum((val_targets - val_targets.mean()) ** 2))
    
    criterion = nn.MSELoss()
    val_loss = criterion(torch.FloatTensor(val_preds), torch.FloatTensor(val_targets)).item()
    
    return {
        'val_loss': val_loss,
        'val_mae': val_mae,
        'val_rmse': val_rmse,
        'val_r2': val_r2
    }


def save_experiment(model, config, history, metrics, X_train):
    """Save all experiment artifacts"""
    
    # 1. Save final model
    model_path = config['experiment_dir'] / 'final_model.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config,
        'input_shape': (X_train.shape[1], X_train.shape[2]),
        'task_type': config['task_type'],
        'val_mae': metrics['val_mae'],
        'val_rmse': metrics['val_rmse'],
        'val_r2': metrics['val_r2'],
        'val_loss': metrics['val_loss'],
        'training_history': history
    }, model_path)
    
    # 2. Save validation results JSON
    val_results = {
        'task_type': config['task_type'],
        'val_loss': float(metrics['val_loss']),
        'val_mae': float(metrics['val_mae']),
        'val_rmse': float(metrics['val_rmse']),
        'val_r2': float(metrics['val_r2'])
    }
    
    with open(config['experiment_dir'] / 'val_results.json', 'w') as f:
        json.dump(val_results, f, indent=2)
    
    # 3. Save configuration JSON
    with open(config['experiment_dir'] / 'config.json', 'w') as f:
        json.dump({k: str(v) for k, v in config.items()}, f, indent=2)
    
    # 4. Save training history plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].axvline(x=history['best_epoch']-1, color='r', linestyle='--', alpha=0.5, label='Best Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (MSE)')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(history['mae'], label='Train MAE', linewidth=2)
    axes[1].plot(history['val_mae'], label='Val MAE', linewidth=2)
    axes[1].axvline(x=history['best_epoch']-1, color='r', linestyle='--', alpha=0.5, label='Best Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].set_title('Mean Absolute Error')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(config['experiment_dir'] / 'training_history.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    # 5. Track in CSV
    experiments_csv = config['save_dir'] / 'experiments.csv'
    
    all_columns = [
        'experiment_name', 'timestamp', 'task_type', 'n_filters', 'dropout_rate',
        'learning_rate', 'batch_size', 'epochs_trained', 'best_epoch', 'notes',
        'val_f1', 'val_accuracy', 'val_precision', 'val_recall',
        'val_mae', 'val_rmse', 'val_r2', 'val_loss'
    ]
    
    experiment_record = {
        'experiment_name': config['experiment_name'],
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'task_type': config['task_type'],
        'n_filters': config['n_filters'],
        'dropout_rate': config['dropout_rate'],
        'learning_rate': config['learning_rate'],
        'batch_size': config['batch_size'],
        'epochs_trained': len(history['loss']),
        'best_epoch': history['best_epoch'],
        'notes': config.get('notes', ''),
        'val_f1': None,
        'val_accuracy': None,
        'val_precision': None,
        'val_recall': None,
        'val_mae': float(metrics['val_mae']),
        'val_rmse': float(metrics['val_rmse']),
        'val_r2': float(metrics['val_r2']),
        'val_loss': float(metrics['val_loss'])
    }
    
    file_exists = experiments_csv.exists()
    with open(experiments_csv, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=all_columns)
        if not file_exists:
            writer.writeheader()
        writer.writerow(experiment_record)


print('✓ Training and evaluation functions defined')

## Run Hyperparameter Search

Loop through all hyperparameter combinations and train models

In [ ]:
# Track all results
all_results = []

print(f'\n{"="*80}')
print(f'STARTING HYPERPARAMETER SEARCH')
print(f'{"="*80}')
print(f'Total experiments: {len(all_combinations)}')
print(f'Model: {MODEL_TYPE}')
print(f'Task: {TASK_TYPE}\n')

for i, combo in enumerate(all_combinations, 1):
    # Create config for this experiment
    combo_dict = dict(zip(param_names, combo))
    
    config = {
        'data_dir': DRIVE_DATA_DIR,
        'save_dir': DRIVE_SAVE_DIR,
        'task_type': TASK_TYPE,
        'model_type': MODEL_TYPE,
        **FIXED_PARAMS,
        **combo_dict,
        'experiment_name': f'{MODEL_TYPE}_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
        'notes': f'Grid search: {combo_dict}'
    }
    
    # Create experiment directory
    experiment_dir = config['save_dir'] / config['experiment_name']
    experiment_dir.mkdir(parents=True, exist_ok=True)
    config['experiment_dir'] = experiment_dir
    
    print(f'\n{"="*80}')
    print(f'EXPERIMENT {i}/{len(all_combinations)}')
    print(f'{"="*80}')
    print(f'Config: {combo_dict}')
    print(f'Folder: {config["experiment_name"]}')
    print()
    
    # Create DataLoaders with current batch_size
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config['batch_size'], 
        shuffle=True, 
        num_workers=2, 
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=config['batch_size'], 
        shuffle=False, 
        num_workers=2, 
        pin_memory=True
    )
    
    # Create model
    if MODEL_TYPE == 'shallow_cnn':
        model = ShallowCNN(
            input_channels=X_train.shape[2],
            sequence_length=X_train.shape[1],
            num_filters=config['n_filters'],
            output_type=config['task_type'],
            dropout_rate=config['dropout_rate']
        )
    elif MODEL_TYPE == 'deep_cnn':
        model = DeepCNN(
            input_channels=X_train.shape[2],
            sequence_length=X_train.shape[1],
            num_filters=config['n_filters'],
            output_type=config['task_type'],
            dropout_rate=config['dropout_rate']
        )
    
    # Train model
    history, model = train_model(model, train_loader, val_loader, config)
    
    # Evaluate model
    metrics = evaluate_model(model, val_loader, config)
    
    # Save all artifacts
    save_experiment(model, config, history, metrics, X_train)
    
    # Store results
    result = {**combo_dict, **metrics, 'experiment_name': config['experiment_name']}
    all_results.append(result)
    
    print(f'\n✓ Experiment {i} complete!')
    print(f'  Val Loss: {metrics["val_loss"]:.4f}')
    print(f'  Val MAE: {metrics["val_mae"]:.4f}')
    print(f'  Val RMSE: {metrics["val_rmse"]:.4f}')
    print(f'  Val R²: {metrics["val_r2"]:.4f}')
    print(f'  Saved to: {experiment_dir}')
    
    # Clear GPU cache
    del model
    torch.cuda.empty_cache()

print(f'\n{"="*80}')
print(f'✅ HYPERPARAMETER SEARCH COMPLETE')
print(f'{"="*80}')
print(f'Total experiments run: {len(all_results)}')

## Summary of Results

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(all_results)

# Sort by validation loss
results_df = results_df.sort_values('val_loss')

print('\nHyperparameter Search Results (sorted by val_loss):')
print('='*80)
print(results_df.to_string(index=False))

print(f'\n\nBest Configuration:')
print('='*80)
best_result = results_df.iloc[0]
for col in results_df.columns:
    print(f'{col}: {best_result[col]}')

# Save summary
summary_path = DRIVE_SAVE_DIR / f'hyperparameter_search_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
results_df.to_csv(summary_path, index=False)
print(f'\n✓ Summary saved to: {summary_path}')